In [ ]:
from pathlib import Path

import pandas as pd


In [3]:
PROJECT_ROOT = Path.cwd().parent

INPUT_FILE = PROJECT_ROOT / "data" / "raw" / "id_list.csv"

print("Project root:", PROJECT_ROOT)
print("File path:", INPUT_FILE)
print("File exists:", INPUT_FILE.exists())

Project root: c:\Users\SuraiyaMohammed\Desktop\StreamVault_Licensing_Analysis
File path: c:\Users\SuraiyaMohammed\Desktop\StreamVault_Licensing_Analysis\data\raw\id_list.csv
File exists: True


### 2. Load the Original Candidate ID Dataset

I started by loading the original candidate ID list supplied for the StreamVault licensing analysis.

At this stage, i am not cleaning, modifying or removing any records. My goal is to preserve the original values so that we can investigate the dataset and account for every candidate row before making any cleaning decisions.

I used the following options when reading the CSV:

- `dtype="string"`: Reads the movie IDs as strings rather than automatically converting them into numbers.
- `keep_default_na=False`: Prevents pandas from automatically interpreting values such as `N/A` as missing values.
- `skip_blank_lines=False`: Ensures that blank lines are not silently skipped during loading.

This allows me to inspect the data in its original form before deciding how to handle missing, duplicated or invalid-looking values.

In [4]:
raw_ids = pd.read_csv(
    INPUT_FILE,
    dtype="string",
    keep_default_na=False,
    skip_blank_lines=False
)

raw_ids.head(10)

,tmdb_id
0,155
1,165
2,242
3,49026
4,8077
5,331
6,680
7,58
8,10138
9,673


### 3. Initial Dataset Overview

Before investigating individual data quality issues, I need to understand the overall structure of the dataset.

This includes checking:
- The total number of candidate records supplied.
- The number of columns available.
- The column names.
- The data types assigned to each column.

This gives me an initial understanding of the dataset and helps me identify whether it has the expected structure.

In [5]:
print("Number of rows:", len(raw_ids))
print("Number of columns:", len(raw_ids.columns))

print("\nColumn names:")
print(raw_ids.columns.tolist())

print("\nData types:")
print(raw_ids.dtypes)

Number of rows: 127
Number of columns: 1

Column names:
['tmdb_id']

Data types:
tmdb_id    string
dtype: object


### 4. Inspecting the Raw Candidate IDs

After establishing the dataset's structure, I will inspect the actual values supplied in the `tmdb_id` column.

This helps me identify unusual entries that may not be immediately visible from the dataset overview, such as blank values, unexpected formats, negative numbers or text values.

I will inspect the beginning and end of the dataset first, without modifying any records.

In [6]:
print("First 15 candidate IDs:")
for value in raw_ids["tmdb_id"].head(15):
    print(repr(value))

print("\nLast 15 candidate IDs:")
for value in raw_ids["tmdb_id"].tail(15):
    print(repr(value))

First 15 candidate IDs:
'155'
'165'
'242'
'49026'
'8077'
'331'
'680'
'58'
'10138'
'673'
'955'
'8587'
'674'
'1573'
'769'

Last 15 candidate IDs:
'604'
'196'
'393'
'218'
'10192'
'534'
'330'
'1895'
'272'
'49051'
'166426'
'157336'
'620'
'296'
'8078'


### 5. Identifying Missing Values and Duplicate IDs

I will now examine the entire candidate ID column to identify:

- Blank or empty entries.
- Duplicate candidate IDs.
- The number of distinct nonblank values.

Duplicate IDs matter because the same film may have been submitted more than once. If I count repeated IDs as separate films, I could unintentionally inflate the number of candidates and distort the licensing analysis.

At this stage, I will identify and quantify these issues without deleting or modifying any records.

In [7]:
ids = raw_ids["tmdb_id"]

# Identify blank or empty entries
blank_mask = ids.str.strip().eq("")

# Identify repeated occurrences, excluding blank entries
duplicate_mask = ids.duplicated(keep="first") & ~blank_mask

print("Total candidate rows:", len(ids))
print("Blank entries:", blank_mask.sum())
print("Duplicate occurrences:", duplicate_mask.sum())
print("Distinct nonblank values:", ids[~blank_mask].nunique())

print("\nDuplicate ID values and their frequencies:")
print(
    ids[ids.duplicated(keep=False) & ~blank_mask]
    .value_counts()
)

Total candidate rows: 127
Blank entries: 1
Duplicate occurrences: 6
Distinct nonblank values: 120

Duplicate ID values and their frequencies:
tmdb_id
155     2
680     2
278     2
603     2
1891    2
550     2
Name: count, dtype: Int64


### 6. Investigating Invalid-Looking and Unexpected ID Formats

I will examine the candidate IDs to identify values that do not follow the expected numeric format of TMDb movie IDs.

In particular, I will check for:
- Negative numbers.
- Zero values.
- IMDb-style identifiers.
- Text entries such as `N/A`.
- Values containing unexpected characters or formats.
- Entries with leading or trailing whitespace.

Identifying these values will help me distinguish between records that may be usable, records that require further validation and records that cannot be queried directly using the TMDb movie endpoint.

I will not classify an ID as valid simply because it contains digits. Actual existence and API recognition will be verified during data acquisition.

In [8]:
# Remove surrounding whitespace for inspection only
cleaned_view = ids.str.strip()

# Identify different ID formats
negative_mask = cleaned_view.str.fullmatch(r"-\d+")
zero_mask = cleaned_view.str.fullmatch(r"0+")
imdb_mask = cleaned_view.str.fullmatch(r"tt\d+", case=False)
numeric_mask = cleaned_view.str.fullmatch(r"\d+")
unexpected_mask = (
    ~blank_mask
    & ~negative_mask
    & ~zero_mask
    & ~imdb_mask
    & ~numeric_mask
)

whitespace_mask = ids.ne(cleaned_view)

print("Negative IDs:", negative_mask.sum())
print("Zero IDs:", zero_mask.sum())
print("IMDb-style IDs:", imdb_mask.sum())
print("Unexpected text or formats:", unexpected_mask.sum())
print("Values with surrounding whitespace:", whitespace_mask.sum())

print("\nNegative IDs:")
print(ids[negative_mask])

print("\nIMDb-style IDs:")
print(ids[imdb_mask])

print("\nUnexpected values:")
print(ids[unexpected_mask])

Negative IDs: 1
Zero IDs: 0
IMDb-style IDs: 1
Unexpected text or formats: 1
Values with surrounding whitespace: 0

Negative IDs:
28    -1
Name: tmdb_id, dtype: string

IMDb-style IDs:
93    tt0468569
Name: tmdb_id, dtype: string

Unexpected values:
109    N/A
Name: tmdb_id, dtype: string


### 7. Initial Data Quality Summary

Based on my initial inspection, I have identified several issues in the candidate ID dataset.

These include blank entries, duplicate IDs and values that do not follow the expected TMDb identifier format.

I will consolidate these findings to establish an initial picture of the dataset's quality. This summary will serve as a reference during subsequent validation and data acquisition.

The findings are preliminary because I have not yet checked whether the numeric identifiers correspond to actual movies in TMDb.

In [9]:
quality_summary = pd.DataFrame({
    "Issue": [
        "Total candidate rows",
        "Blank entries",
        "Duplicate occurrences",
        "Distinct nonblank values",
        "Negative IDs",
        "Zero IDs",
        "IMDb-style IDs",
        "Unexpected text or formats",
        "Values with surrounding whitespace"
    ],
    "Count": [
        len(ids),
        blank_mask.sum(),
        duplicate_mask.sum(),
        ids[~blank_mask].nunique(),
        negative_mask.sum(),
        zero_mask.sum(),
        imdb_mask.sum(),
        unexpected_mask.sum(),
        whitespace_mask.sum()
    ]
})

quality_summary

,Issue,Count
0,Total candidate rows,127
1,Blank entries,1
2,Duplicate occurrences,6
3,Distinct nonblank values,120
4,Negative IDs,1
5,Zero IDs,0
6,IMDb-style IDs,1
7,Unexpected text or formats,1
8,Values with surrounding whitespace,0


### 8. Accounting for Every Original Candidate Row

I will now examine how the 127 original records are distributed across different categories.

The purpose is to establish a clear record of the dataset before any records are excluded from further analysis.

I will distinguish between blank entries, duplicate occurrences, incorrectly formatted identifiers and unique values that still require validation.

This accounting will help me avoid losing records silently and provide a foundation for the candidate ID validation process.

In [10]:
print("Total original rows:", len(ids))
print("Blank entries:", blank_mask.sum())
print("Duplicate occurrences:", duplicate_mask.sum())
print("Unique nonblank values:", ids[~blank_mask].nunique())

print(
    "\nReconciliation:",
    blank_mask.sum() + duplicate_mask.sum() + ids[~blank_mask].nunique()
)

Total original rows: 127
Blank entries: 1
Duplicate occurrences: 6
Unique nonblank values: 120

Reconciliation: 127


### 9. Creating a Record-Level Investigation Table

After reconciling the original records, I will create an investigation table that preserves each candidate's original row position and ID value.

I will also attach preliminary classifications to help identify blank entries, duplicate occurrences, negative IDs, IMDb-style identifiers and unexpected formats.

This allows me to trace individual records throughout the investigation without modifying the original dataset.

These classifications are preliminary and will be refined during TMDb validation.

In [11]:
investigation = raw_ids.copy()

investigation.insert(
    0,
    "source_row",
    range(1, len(investigation) + 1)
)

investigation["id_format"] = "numeric-looking"

investigation.loc[blank_mask, "id_format"] = "blank"
investigation.loc[negative_mask, "id_format"] = "negative"
investigation.loc[zero_mask, "id_format"] = "zero"
investigation.loc[imdb_mask, "id_format"] = "imdb_style"
investigation.loc[unexpected_mask, "id_format"] = "unexpected"

investigation["is_duplicate"] = ids.duplicated(keep="first") & ~blank_mask

investigation.head(15)

,source_row,tmdb_id,id_format,is_duplicate
0,1,155,numeric-looking,False
1,2,165,numeric-looking,False
2,3,242,numeric-looking,False
3,4,49026,numeric-looking,False
4,5,8077,numeric-looking,False
5,6,331,numeric-looking,False
6,7,680,numeric-looking,False
7,8,58,numeric-looking,False
8,9,10138,numeric-looking,False
9,10,673,numeric-looking,False


### 10. Reviewing Records Requiring Attention

I will now isolate the records identified during my initial investigation as requiring further attention.

This includes:
- Blank or incorrectly formatted identifiers.
- Negative and IMDb-style identifiers.
- Duplicate candidate IDs.

I will review these records together to understand the nature and extent of the issues before proceeding to external validation.

No records will be removed at this stage.

In [12]:
# Records with unusual ID formats
print("Records with unusual ID formats:")

unusual_records = investigation[
    investigation["id_format"] != "numeric-looking"
]

display(unusual_records)

# Duplicate records
print("\nDuplicate candidate records:")

duplicate_records = investigation[
    investigation["is_duplicate"]
]

display(duplicate_records)

Records with unusual ID formats:


,source_row,tmdb_id,id_format,is_duplicate
28,29,-1,negative,False
67,68,,blank,False
93,94,tt0468569,imdb_style,False
109,110,N/A,unexpected,False



Duplicate candidate records:


,source_row,tmdb_id,id_format,is_duplicate
41,42,603,numeric-looking,True
58,59,278,numeric-looking,True
88,89,1891,numeric-looking,True
89,90,155,numeric-looking,True
92,93,550,numeric-looking,True
100,101,680,numeric-looking,True
